# Lab 06 — TF-IDF + Logistic baseline

Трек: **A (класифікація)**. Мета: побудувати й порівняти 2 відтворювані baseline-варіанти, оцінити `accuracy + macro-F1`, показати confusion matrix, top features і зробити мінімальний error analysis.


## 1) Install deps

In [1]:
!pip -q install -r ../requirements.txt

## 2) Data access + load split (from Lab5)

In [2]:
from pathlib import Path
import json
import re
import sys
from collections import Counter

import pandas as pd

LAB6_ROOT = Path('..').resolve()
LAB2_ROOT = (LAB6_ROOT.parent / 'project_lab2').resolve()
LAB5_ROOT = (LAB6_ROOT.parent / 'project_lab5').resolve()

sys.path.insert(0, str(LAB6_ROOT))

from src.classification_baseline import (
    BaselineConfig,
    confusion_table,
    evaluate_baseline,
    load_split_ids,
    subset_by_ids,
    top_features_per_class,
)

processed_path = LAB2_ROOT / 'data' / 'processed_v2' / 'processed_v2.csv'
df = pd.read_csv(processed_path)
df['text'] = df['text'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

train_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_train_ids.txt')
val_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_val_ids.txt')
test_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_test_ids.txt')

df_train = subset_by_ids(df, train_ids, id_col='text_id')
df_val = subset_by_ids(df, val_ids, id_col='text_id')
df_test = subset_by_ids(df, test_ids, id_col='text_id')

print('processed_v2:', processed_path)
print('Shapes:', {'train': len(df_train), 'val': len(df_val), 'test': len(df_test), 'total': len(df)})
print('Class dist train:')
print(df_train['label'].value_counts())


processed_v2: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
Shapes: {'train': 800, 'val': 100, 'test': 100, 'total': 1000}
Class dist train:
label
Question / Request for Help      160
Gratitude / Positive Feedback    160
Complaint / Dissatisfaction      160
Neutral Comment                  160
Suggestion / Idea                160
Name: count, dtype: int64


## 3) Baseline 1 — TF-IDF word(1,1) + Logistic Regression

In [3]:
cfg1 = BaselineConfig(
    name='baseline_1_word_1_1',
    ngram_range=(1, 1),
    class_weight=None,
    max_iter=500,
    random_state=42,
)

res1 = evaluate_baseline(cfg1, df_train, df_val, df_test)

print('Baseline 1')
print(f"VAL  acc={res1['val_accuracy']:.4f}, macroF1={res1['val_macro_f1']:.4f}")
print(f"TEST acc={res1['test_accuracy']:.4f}, macroF1={res1['test_macro_f1']:.4f}")
print() 
print('VAL classification_report:')
print(res1['val_report'])
print()
print('TEST classification_report:')
print(res1['test_report'])


Baseline 1
VAL  acc=0.5700, macroF1=0.5730
TEST acc=0.6800, macroF1=0.6786

VAL classification_report:
                               precision    recall  f1-score   support

  Complaint / Dissatisfaction     0.4643    0.6500    0.5417        20
Gratitude / Positive Feedback     0.8000    0.6000    0.6857        20
              Neutral Comment     0.4444    0.4000    0.4211        20
  Question / Request for Help     0.6842    0.6500    0.6667        20
            Suggestion / Idea     0.5500    0.5500    0.5500        20

                     accuracy                         0.5700       100
                    macro avg     0.5886    0.5700    0.5730       100
                 weighted avg     0.5886    0.5700    0.5730       100


TEST classification_report:
                               precision    recall  f1-score   support

  Complaint / Dissatisfaction     0.5769    0.7500    0.6522        20
Gratitude / Positive Feedback     0.6957    0.8000    0.7442        20
            

## 4) Baseline 2 — TF-IDF word(1,2) + Logistic Regression

In [4]:
cfg2 = BaselineConfig(
    name='baseline_2_word_1_2',
    ngram_range=(1, 2),
    class_weight=None,
    max_iter=500,
    random_state=42,
)

res2 = evaluate_baseline(cfg2, df_train, df_val, df_test)

print('Baseline 2')
print(f"VAL  acc={res2['val_accuracy']:.4f}, macroF1={res2['val_macro_f1']:.4f}")
print(f"TEST acc={res2['test_accuracy']:.4f}, macroF1={res2['test_macro_f1']:.4f}")
print()
print('VAL classification_report:')
print(res2['val_report'])
print()
print('TEST classification_report:')
print(res2['test_report'])


Baseline 2
VAL  acc=0.5700, macroF1=0.5741
TEST acc=0.6800, macroF1=0.6786

VAL classification_report:
                               precision    recall  f1-score   support

  Complaint / Dissatisfaction     0.4516    0.7000    0.5490        20
Gratitude / Positive Feedback     0.7857    0.5500    0.6471        20
              Neutral Comment     0.4737    0.4500    0.4615        20
  Question / Request for Help     0.7059    0.6000    0.6486        20
            Suggestion / Idea     0.5789    0.5500    0.5641        20

                     accuracy                         0.5700       100
                    macro avg     0.5992    0.5700    0.5741       100
                 weighted avg     0.5992    0.5700    0.5741       100


TEST classification_report:
                               precision    recall  f1-score   support

  Complaint / Dissatisfaction     0.5769    0.7500    0.6522        20
Gratitude / Positive Feedback     0.6957    0.8000    0.7442        20
            

## 5) Metrics comparison + winner

In [5]:
metrics_df = pd.DataFrame([
    {
        'baseline': res1['name'],
        'val_accuracy': res1['val_accuracy'],
        'val_macro_f1': res1['val_macro_f1'],
        'test_accuracy': res1['test_accuracy'],
        'test_macro_f1': res1['test_macro_f1'],
    },
    {
        'baseline': res2['name'],
        'val_accuracy': res2['val_accuracy'],
        'val_macro_f1': res2['val_macro_f1'],
        'test_accuracy': res2['test_accuracy'],
        'test_macro_f1': res2['test_macro_f1'],
    },
]).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)

best_name = metrics_df.iloc[0]['baseline']
best = res1 if best_name == res1['name'] else res2

print(metrics_df)
print()
print('Winner by val_macro_f1:', best_name)


              baseline  val_accuracy  val_macro_f1  test_accuracy  \
0  baseline_2_word_1_2          0.57      0.574074           0.68   
1  baseline_1_word_1_1          0.57      0.573020           0.68   

   test_macro_f1  
0       0.678608  
1       0.678608  

Winner by val_macro_f1: baseline_2_word_1_2


## 6) Confusion matrix (test)

In [6]:
labels = sorted(df_train['label'].astype(str).unique().tolist())
cm_df = confusion_table(best['y_test'], best['pred_test'], labels=labels)
print('Confusion matrix for', best['name'])
print(cm_df)


Confusion matrix for baseline_2_word_1_2
                                     pred::Complaint / Dissatisfaction  \
gold::Complaint / Dissatisfaction                                   15   
gold::Gratitude / Positive Feedback                                  1   
gold::Neutral Comment                                                3   
gold::Question / Request for Help                                    4   
gold::Suggestion / Idea                                              3   

                                     pred::Gratitude / Positive Feedback  \
gold::Complaint / Dissatisfaction                                      2   
gold::Gratitude / Positive Feedback                                   16   
gold::Neutral Comment                                                  4   
gold::Question / Request for Help                                      0   
gold::Suggestion / Idea                                                1   

                                     pred::Neutral Commen

## 7) Top features (LogReg interpretability)

In [7]:
top_feats = top_features_per_class(best['pipeline'], top_n=10)

for cls, block in top_feats.items():
    print()
    print('=' * 80)
    print('CLASS:', cls)
    print('Top + features:')
    for feat, w in block['top_positive']:
        print(f'  + {feat:<30} {w:.4f}')



CLASS: Complaint / Dissatisfaction
Top + features:
  + не                             2.8499
  + що                             1.0982
  + нічого                         0.9426
  + чергу                          0.9236
  + сказали                        0.8565
  + так                            0.7959
  + відношення                     0.7786
  + немає                          0.7284
  + потім                          0.7243
  + неможливо                      0.7151

CLASS: Gratitude / Positive Feedback
Top + features:
  + дуже                           2.2738
  + дякую                          2.0052
  + швидко                         1.2437
  + та                             1.1159
  + задоволений                    1.1014
  + якісно                         1.0391
  + викладачі                      1.0319
  + рекомендую                     0.9876
  + консультацію                   0.9539
  + відмінний                      0.8540

CLASS: Neutral Comment
Top + features:
  + будівля   

## 8) Minimal error analysis (10 examples)

In [8]:
pred_test = pd.Series(best['pred_test'], index=df_test.index)

a = df_test.copy()
a['pred_label'] = pred_test.values
errors = a[a['label'].astype(str) != a['pred_label'].astype(str)].copy().reset_index(drop=True)


def categorize_error(text: str) -> str:
    t = str(text)
    tl = t.lower()
    wc = len(t.split())
    if wc <= 4:
        return 'короткий текст / бракує контексту'
    if re.search(r'[A-Za-z]', t):
        return 'трансліт / латинка / шум'
    if '?' in t and ('не' in tl or 'жах' in tl or 'поган' in tl):
        return 'overlap: питання + скарга'
    if any(x in tl for x in ['дякую', 'рекомендую', 'жах', 'проблем', 'погано', 'чудово']):
        return 'overlap класів / змішаний намір'
    if wc > 35:
        return 'довгий multi-intent текст'
    return 'рідкісна лексика / неоднозначність'


def build_comment(cat: str) -> str:
    if cat == 'короткий текст / бракує контексту':
        return 'Уривок занадто короткий, сигналу замало для надійного класу.'
    if cat == 'трансліт / латинка / шум':
        return 'Є латинка/шум, модель гірше узагальнює такі токени.'
    if cat == 'overlap: питання + скарга':
        return 'Текст одночасно запитує і виражає негатив, межа класів розмита.'
    if cat == 'overlap класів / змішаний намір':
        return 'У тексті кілька намірів, модель обирає домінантний не так, як gold.'
    if cat == 'довгий multi-intent текст':
        return 'Довгий текст з кількома тезами, baseline втрачає головний намір.'
    return 'Ймовірна рідкісна лексика або неоднозначна розмітка.'

errors['error_category'] = errors['text'].apply(categorize_error)
errors['comment'] = errors['error_category'].apply(build_comment)

sample_errors = errors.head(10).copy()
print('Total test errors:', len(errors))
print(sample_errors[['text_id', 'label', 'pred_label', 'error_category', 'comment']])

# Save optional error cases file
err_out = LAB6_ROOT / 'tests' / 'error_cases_lab6.jsonl'
rows = []
for _, r in sample_errors.iterrows():
    rows.append({
        'text_id': int(r['text_id']),
        'text': str(r['text']),
        'gold_label': str(r['label']),
        'pred_label': str(r['pred_label']),
        'error_category': str(r['error_category']),
        'comment': str(r['comment']),
    })
with err_out.open('w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print('Saved:', err_out)

error_category_counts = errors['error_category'].value_counts().to_dict()
print('Error categories:', error_category_counts)


Total test errors: 32
    text_id                        label                     pred_label  \
0  10004407            Suggestion / Idea                Neutral Comment   
1     11219              Neutral Comment  Gratitude / Positive Feedback   
2     16507            Suggestion / Idea    Complaint / Dissatisfaction   
3     14094              Neutral Comment    Complaint / Dissatisfaction   
4     16431              Neutral Comment              Suggestion / Idea   
5  10004733  Question / Request for Help    Complaint / Dissatisfaction   
6       687              Neutral Comment    Complaint / Dissatisfaction   
7     10425            Suggestion / Idea                Neutral Comment   
8      4213              Neutral Comment  Gratitude / Positive Feedback   
9     14545              Neutral Comment    Question / Request for Help   

                       error_category  \
0           overlap: питання + скарга   
1  рідкісна лексика / неоднозначність   
2  рідкісна лексика / неодноз

## 9) Generate docs/audit_summary_lab6.md + dataset_card + lab06 README

In [9]:
docs_dir = LAB6_ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)

audit_path = docs_dir / 'audit_summary_lab6.md'
card_path = docs_dir / 'dataset_card.md'
readme_path = LAB6_ROOT / 'labs' / 'lab06' / 'README.md'

winner = metrics_df.iloc[0]
runner = metrics_df.iloc[1]

common_error_types = list(error_category_counts.keys())[:3]
while len(common_error_types) < 3:
    common_error_types.append('n/a')

audit_lines = []
audit_lines.append('# Audit summary — Lab6')
audit_lines.append('')
audit_lines.append('1) Підзадача: multi-class classification (Track A) для категорій повідомлень UAReviews.')
audit_lines.append('2) Split: використано Lab5 split train/val/test = 800/100/100, seed=42.')
audit_lines.append(f"3) Baseline 1 ({res1['name']}): accuracy={res1['test_accuracy']:.4f}, macro-F1={res1['test_macro_f1']:.4f} (test).")
audit_lines.append(f"4) Baseline 2 ({res2['name']}): accuracy={res2['test_accuracy']:.4f}, macro-F1={res2['test_macro_f1']:.4f} (test).")
audit_lines.append(
    f"5) Winner: {winner['baseline']} (val macro-F1={winner['val_macro_f1']:.4f}); "
    f"delta vs other on test macro-F1 = {winner['test_macro_f1'] - runner['test_macro_f1']:+.4f}."
)
audit_lines.append(f"6) Топ-3 категорії помилок: {common_error_types[0]}, {common_error_types[1]}, {common_error_types[2]}.")
audit_lines.append('7) Далі: дедуп/near-dup фільтрація, розширення n-gram/char-features, ревізія прикордонних label-пар.')
audit_path.write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')

card_lines = []
card_lines.append('# Dataset Card — Lab6 update')
card_lines.append('')
card_lines.append('## Classification baseline')
card_lines.append('- Model: TF-IDF + Logistic Regression (2 baseline variants).')
card_lines.append('- Features: processed_v2 text, word n-grams (1,1) and (1,2).')
card_lines.append('')
card_lines.append('## Main risks')
card_lines.append('- overlap класів та змішані наміри в одному тексті')
card_lines.append('- noisy labels / неоднозначність gold')
card_lines.append('- translit/slang та рідкісна лексика')
card_lines.append('- domain drift на нових джерелах')
card_path.write_text('\n'.join(card_lines) + '\n', encoding='utf-8')

# lab06 README (5 required points)
rd = []
rd.append('# LPNU NLP — Lab 06 (TF-IDF + Logistic baseline)')
rd.append('')
rd.append('1. Напрям: A (класифікація).')
rd.append('2. Підзадача: класифікація українських текстів UAReviews на 5 категорій.')
rd.append(f"3. Порівняні baseline-и: {res1['name']} та {res2['name']}.")
rd.append(
    f"4. Основні цифри (test): "
    f"{res1['name']} acc={res1['test_accuracy']:.4f}, macro-F1={res1['test_macro_f1']:.4f}; "
    f"{res2['name']} acc={res2['test_accuracy']:.4f}, macro-F1={res2['test_macro_f1']:.4f}."
)
rd.append(
    '5. Error analysis: найчастіше трапляються overlap класів, короткі тексти та noisy/translit кейси; '
    'далі варто покращити розмітку прикордонних випадків і додати більш стійкі фічі.'
)
readme_path.write_text('\n'.join(rd) + '\n', encoding='utf-8')

print('Saved:', audit_path)
print('Saved:', card_path)
print('Saved:', readme_path)


Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\docs\audit_summary_lab6.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\docs\dataset_card.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\labs\lab06\README.md


In [10]:
print('Lab6 artifacts ready:')
print('-', LAB6_ROOT / 'src' / 'classification_baseline.py')
print('-', LAB6_ROOT / 'notebooks' / 'lab6_tfidf_logistic_baseline.ipynb')
print('-', LAB6_ROOT / 'docs' / 'audit_summary_lab6.md')
print('-', LAB6_ROOT / 'docs' / 'dataset_card.md')
print('-', LAB6_ROOT / 'tests' / 'error_cases_lab6.jsonl')
print('-', LAB6_ROOT / 'labs' / 'lab06' / 'README.md')


Lab6 artifacts ready:
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\src\classification_baseline.py
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\notebooks\lab6_tfidf_logistic_baseline.ipynb
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\docs\audit_summary_lab6.md
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\docs\dataset_card.md
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\tests\error_cases_lab6.jsonl
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab6\labs\lab06\README.md
